In [ ]:
# Import library 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kardemumma as kdm



In [ ]:
# Read files 
htipick_data = pd.read_csv('data/DA4000/hitpick_P1-8_v2(Sheet1).csv', sep=';') 

In [ ]:
# Create mapping pandas DataFrame with 3 columns: POS_ID (1-96), Row (A-H), Col (1-12)
rows = list("ABCDEFGH")
cols = list(range(1, 13))
# Create a list of (Row, Col) tuples in plate order
wells = [(row, col) for row in rows for col in cols]  # 8*12 = 96
mapping_df_row = pd.DataFrame({
    "POS_ID": range(1, 97),
    "Row": [row for row, col in wells],
    "Col": [col for row, col in wells]
})


In [ ]:
# Create mapping df sorted by Col first (so Col changes fastest)
wells_col = [(row, col) for col in cols for row in rows]  # Col changes fastest, then Row
mapping_df_col = pd.DataFrame({
    "POS_ID": range(1, 97),
    "Row": [row for row, col in wells_col],
    "Col": [col for row, col in wells_col]
})


In [ ]:
# Remove any completely empty rows (all NaN) instead of by 'Unnamed: 0' column, to avoid KeyError
htipick_data = htipick_data.dropna(how='all').copy()

htipick_data.head()


In [ ]:
# merge hitpick data with mapping_df_row
htipick_data = pd.merge(htipick_data, mapping_df_col, left_on='dest_pos', right_on='POS_ID', how='outer')

# Sort by source_plate, Row and Col
htipick_data = htipick_data.sort_values(by=['dest_plate', 'Row', 'Col'])

# Select columns; source_plate, Row, Col, Disease
htipick_data = htipick_data[['dest_plate', 'POS_ID', 'Row', 'Col', 'Disease']]

In [ ]:
htipick_data.head()

In [ ]:
# Create ideal_df with 3 dest_plate (1 to 38), Row and Col like mapping_df_col (Col changes fastest)
plates = range(1, 39)
wells_ideal = [(row, col) for col in cols for row in rows]  # 96 wells per plate, Col changes fastest
ideal_df = pd.DataFrame([
    {'dest_plate': plate, 'Row': row, 'Col': col}
    for plate in plates
    for row, col in wells_ideal
])

# Sort by dest_plate, Row and Col
ideal_df = ideal_df.sort_values(by=['dest_plate', 'Row', 'Col'])
ideal_df

In [ ]:
# Join ideal_df with htipick_data
ideal_df = pd.merge(ideal_df, htipick_data, on=['dest_plate', 'Row', 'Col'], how='left')

# Fill in Disease column with 'PlasmaPool' for empty wells
ideal_df['Disease'] = ideal_df['Disease'].fillna('PlasmaPool')
ideal_df.head()

ideal_df.to_csv('data/DA4000/hitpick_P1-8_v2_map.csv', index=False)